# AMPS II · Week 11 — Lab
## Natural convection: Boussinesq flow, mixed $(\mathbf u,p,T)$, Newton with Rayleigh-number continuation

| | |
|---|---|
| **Lecture sections** | Part A (Boussinesq, scalings, the benchmark) · Part B (one mixed space, Newton, continuation, Picard) · Part C (verification, the Nusselt number) · Part D (heated from below, LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | estimate the boundary layer and count your cells in it · solve the de Vahl Davis cavity at $Ra=10^3$ and compute $Nu$ two ways · reach $Ra=10^6$ by continuation and see a direct start fail · compare Picard with Newton · judge an LLM's cavity script |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The model module `bouss.py` is written once with `%%writefile` (§1). All cells run with `-np 2`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model module  *(lecture §A–D)*
`bouss.py`: `box_mesh(N, grade=...)` (graded towards the walls), `Convection` (Taylor–Hood + P2 temperature in one mixed space; `heated="side"` for the cavity, `"below"` for Rayleigh–Bénard;
`.solve(Ra)`, `.continuation([...])`, `.nusselt()` from the wall gradient, `.nusselt_flux()` as a consistent flux, `.peak(comp, x0=..)`/`.peak(comp, y0=..)`, `.probe`, `.umax`, `.history`,
manufactured solutions with `mms=`), `Picard` (the segregated iteration) and the benchmark table `DVD`. Nondimensional with $L$ and $\kappa/L$, as the benchmark. Run the cell.

In [ ]:
%%writefile /tmp/fnx/bouss.py
"""bouss.py — Boussinesq natural convection in a box, DOLFINx 0.11 (AMPS II, Week 11).

Nondimensional with L and the thermal velocity kappa/L (the scaling of the de Vahl Davis benchmark):
    (u.grad) u = -grad p + Pr lap u + Pr Ra T e_y,   div u = 0,   u.grad T = lap T
Taylor-Hood (P2/P1) for (u, p) and P2 for T in one mixed space; Newton (monolithic) or Picard (segregated)."""
import gc
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import NonlinearProblem
import dolfinx.fem.petsc as fp

comm = MPI.COMM_WORLD
# PETSc objects must be destroyed collectively: no automatic garbage collection, collect() at the same point on all ranks
gc.disable()
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
NEWTON = dict(LU, snes_type="newtonls", snes_linesearch_type="none", snes_rtol=1e-10, snes_atol=1e-9, snes_stol=0.0, snes_max_it=25)

def box_mesh(N, L=1.0, H=1.0, grade=0.0, Ny=None):
    """[0, L] x [0, H] with N x Ny triangles; grade in [0, 1): cells clustered towards all walls (s -> s - grade sin(2 pi s)/(2 pi))"""
    msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [N, Ny or N], mesh.CellType.triangle)
    if grade > 0:
        x = msh.geometry.x
        for i, Li in ((0, L), (1, H)):
            s = x[:, i] / Li; x[:, i] = Li * (s - grade * np.sin(2 * np.pi * s) / (2 * np.pi))
    return msh

def _box(msh):
    x = msh.geometry.x
    return comm.allreduce(x[:, 0].max(), op=MPI.MAX), comm.allreduce(x[:, 1].max(), op=MPI.MAX)

class Convection:
    count = 0
    def __init__(self, msh, Ra=1e3, Pr=0.71, heated="side", deg=2, mms=None, guess="conduction", roll=0.0):
        """heated "side": T = 1 at x = 0, T = 0 at x = L, top and bottom adiabatic (de Vahl Davis);
        heated "below": T = 1 at y = 0, T = 0 at y = H, sides adiabatic (Rayleigh-Benard). No slip everywhere; p = 0 at (0, 0).
        mms: dict of UFL expressions u, p, T of the coordinates -> source terms and Dirichlet data from them (all walls).
        guess: "conduction" (linear T, u = 0) or "zero"; roll: amplitude of an added convection roll (u and T), see reset()."""
        collect()
        self.msh = msh; c = msh.basix_cell(); self.deg = deg; self.heated = heated
        self.L, self.H = _box(msh)
        el = basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, deg, shape=(2,)), basix.ufl.element("Lagrange", c, deg - 1),
                                      basix.ufl.element("Lagrange", c, deg)])
        W = self.W = fem.functionspace(msh, el); self.w = fem.Function(W)
        self.Ra, self.Pr = fem.Constant(msh, float(Ra)), fem.Constant(msh, float(Pr))
        self.ey = ufl.as_vector([0.0, 1.0]); self.n = ufl.FacetNormal(msh)
        u, p, T = ufl.split(self.w); v, q, s = ufl.TestFunctions(W)
        self.F = (ufl.inner(ufl.grad(u) * u, v) + self.Pr * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)
                  - self.Pr * self.Ra * T * ufl.dot(self.ey, v) + ufl.dot(u, ufl.grad(T)) * s + ufl.dot(ufl.grad(T), ufl.grad(s))) * ufl.dx
        self.mms = mms
        if mms is not None:
            ue, pe, Te = mms["u"], mms["p"], mms["T"]
            f = ufl.grad(ue) * ue - self.Pr * ufl.div(ufl.grad(ue)) + ufl.grad(pe) - self.Pr * self.Ra * Te * self.ey
            g = ufl.dot(ue, ufl.grad(Te)) - ufl.div(ufl.grad(Te))
            self.F -= (ufl.dot(f, v) + g * s) * ufl.dx
        self.V, _ = W.sub(0).collapse(); self.Q, _ = W.sub(1).collapse(); self.S, self.mapS = W.sub(2).collapse()
        self.bcs = self._bcs()
        Convection.count += 1
        self.pb = NonlinearProblem(self.F, self.w, bcs=self.bcs, petsc_options_prefix=f"cv{Convection.count}_", petsc_options=NEWTON)
        self.pb.solver.setConvergenceHistory(reset=True)
        self.reset(guess, roll)

    def _bcs(self):
        W, fdim = self.W, 1
        allb = mesh.locate_entities_boundary(self.msh, fdim, lambda x: np.full(x.shape[1], True))
        pd = fem.locate_dofs_geometrical((W.sub(1), self.Q), lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0))
        if self.mms is not None:                                        # exact data on every wall, exact pressure at the corner
            uD, pD, TD = fem.Function(self.V), fem.Function(self.Q), fem.Function(self.S)
            for fun, e in ((uD, self.mms["u"]), (pD, self.mms["p"]), (TD, self.mms["T"])):
                fun.interpolate(fem.Expression(e, fun.function_space.element.interpolation_points))
            self.ax = None
            return [fem.dirichletbc(uD, fem.locate_dofs_topological((W.sub(0), self.V), fdim, allb), W.sub(0)),
                    fem.dirichletbc(pD, pd, W.sub(1)),
                    fem.dirichletbc(TD, fem.locate_dofs_topological((W.sub(2), self.S), fdim, allb), W.sub(2))]
        ax, top = (0, self.L) if self.heated == "side" else (1, self.H)
        hot = mesh.locate_entities_boundary(self.msh, fdim, lambda x: np.isclose(x[ax], 0))
        cold = mesh.locate_entities_boundary(self.msh, fdim, lambda x: np.isclose(x[ax], top))
        self.hot, self.cold, self.ax = hot, cold, ax
        tags = np.concatenate([hot, cold]); vals = np.r_[np.full(len(hot), 1), np.full(len(cold), 2)].astype(np.int32); o = np.argsort(tags)
        self.ds = ufl.Measure("ds", domain=self.msh, subdomain_data=mesh.meshtags(self.msh, fdim, tags[o], vals[o]))
        Th = fem.Function(self.S); Th.x.array[:] = 1.0
        return [fem.dirichletbc(fem.Function(self.V), fem.locate_dofs_topological((W.sub(0), self.V), fdim, allb), W.sub(0)),
                fem.dirichletbc(fem.Function(self.Q), pd, W.sub(1)),
                fem.dirichletbc(Th, fem.locate_dofs_topological((W.sub(2), self.S), fdim, hot), W.sub(2)),
                fem.dirichletbc(fem.Function(self.S), fem.locate_dofs_topological((W.sub(2), self.S), fdim, cold), W.sub(2))]

    def reset(self, guess="conduction", roll=0.0):
        """initial guess: conduction (T linear between the walls, u = 0) or zero; plus a roll filling the box:
        u = 10 roll * curl(sin^2(pi x/L) sin^2(pi y/H)) / pi, T += 0.3 roll cos(pi x/L) sin(pi y/H) (warm fluid rising at x = 0)"""
        self.w.x.array[:] = 0.0
        if guess == "conduction" and self.mms is None:
            tf = fem.Function(self.S); top = self.L if self.ax == 0 else self.H; ax = self.ax
            tf.interpolate(lambda x: 1 - x[ax] / top); self.w.x.array[self.mapS] = tf.x.array
        if roll:
            a, L, H = 10.0 * roll, self.L, self.H
            Vc, mV = self.W.sub(0).collapse(); uf = fem.Function(Vc)
            uf.interpolate(lambda x: np.vstack([-a * np.sin(np.pi * x[0] / L) ** 2 * np.sin(2 * np.pi * x[1] / H),
                                                a * np.sin(2 * np.pi * x[0] / L) * np.sin(np.pi * x[1] / H) ** 2]))
            self.w.x.array[mV] = uf.x.array
            tf = fem.Function(self.S); tf.interpolate(lambda x: 0.3 * roll * np.cos(np.pi * x[0] / L) * np.sin(np.pi * x[1] / H))
            self.w.x.array[self.mapS] += tf.x.array
        self.w.x.scatter_forward()

    def solve(self, Ra=None):
        """Newton on the mixed (u, p, T) system from the current state; returns (SNES reason, iterations)"""
        if Ra is not None:
            self.Ra.value = Ra
        self.pb.solve(); r = self.pb.solver
        return int(r.getConvergedReason()), int(r.getIterationNumber())

    def history(self):
        """residual norms of the last Newton solve"""
        return np.array(self.pb.solver.getConvergenceHistory()[0])

    def continuation(self, Ras):
        """Newton at each Ra, starting from the previous solution; returns [(Ra, reason, iterations)]"""
        return [(Ra, *self.solve(Ra)) for Ra in Ras]

    # ── quantities ──────────────────────────────────────────────────
    def nusselt(self):
        """(Nu_hot, Nu_cold) from the wall gradient: mean heat flux into the fluid at the hot wall / out of it at the cold wall"""
        T = ufl.split(self.w)[2]
        f = lambda t: comm.allreduce(fem.assemble_scalar(fem.form(ufl.dot(ufl.grad(T), self.n) * self.ds(t))), op=MPI.SUM)
        wall = self.H if self.ax == 0 else self.L
        return f(1) / wall, -f(2) / wall

    def nusselt_flux(self):
        """(Nu_hot, Nu_cold) as a consistent (residual) flux: the weak energy equation tested with a function that is 1 on one wall"""
        u, p, T = ufl.split(self.w); out = []
        for facets in (self.hot, self.cold):
            z = fem.Function(self.S); z.x.array[fem.locate_dofs_topological(self.S, 1, facets)] = 1.0; z.x.scatter_forward()
            out.append(comm.allreduce(fem.assemble_scalar(fem.form((ufl.dot(u, ufl.grad(T)) * z + ufl.dot(ufl.grad(T), ufl.grad(z))) * ufl.dx)), op=MPI.SUM))
        wall = self.H if self.ax == 0 else self.L
        return out[0] / wall, -out[1] / wall

    def errors(self):
        """L2 errors of u, p, T and H1-seminorm errors of u, T against the manufactured solution"""
        u, p, T = ufl.split(self.w); m = self.mms
        a = lambda e: np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM))
        du, dT = u - m["u"], T - m["T"]
        return dict(u=a(ufl.inner(du, du)), p=a((p - m["p"]) ** 2), T=a(dT ** 2), gu=a(ufl.inner(ufl.grad(du), ufl.grad(du))), gT=a(ufl.inner(ufl.grad(dT), ufl.grad(dT))))

    def probe(self, comp, P):
        """values at points P (n x 3) of comp 0, 1 = u, v; 2 = T; 3 = p — every rank gets all values"""
        fun = self.w.sub({0: 0, 1: 0, 2: 2, 3: 1}[comp]).collapse()
        nloc = self.msh.topology.index_map(2).size_local
        cand = geometry.compute_colliding_cells(self.msh, geometry.compute_collisions_points(geometry.bb_tree(self.msh, 2), P), P)
        out, cnt = np.zeros(len(P)), np.zeros(len(P)); idx, cells = [], []
        for j in range(len(P)):
            own = [c for c in cand.links(j) if c < nloc]
            if own:
                idx.append(j); cells.append(own[0])
        if idx:
            val = np.asarray(fun.eval(P[idx], np.array(cells, dtype=np.int32))).reshape(len(idx), -1)
            out[idx] = val[:, comp] if comp < 2 else val[:, 0]; cnt[idx] = 1
        out = comm.allreduce(out, op=MPI.SUM); cnt = comm.allreduce(cnt, op=MPI.SUM)
        return out / np.maximum(cnt, 1)

    def line(self, comp, x0=None, y0=None, n=401):
        t = np.linspace(0, 1, n); P = np.zeros((n, 3))
        if x0 is not None:
            P[:, 0], P[:, 1] = x0, t * self.H
        else:
            P[:, 0], P[:, 1] = t * self.L, y0
        return t, self.probe(comp, P)

    def peak(self, comp, x0=None, y0=None, n=2001):
        """maximum of a component along a line and its position (parabola through the three best samples)"""
        t, v = self.line(comp, x0=x0, y0=y0, n=n); i = int(np.argmax(v))
        if 0 < i < n - 1:
            a, b, c = v[i - 1], v[i], v[i + 1]; s = 0.5 * (a - c) / (a - 2 * b + c)
            return b - 0.25 * (a - c) * s, t[i] + s * (t[1] - t[0])
        return v[i], t[i]

    def umax(self):
        """max |u| over the velocity dofs"""
        a = self.w.x.array[self.W.sub(0).collapse()[1]]
        return float(comm.allreduce(np.abs(a).max() if len(a) else 0.0, op=MPI.MAX))

class Picard:
    """segregated iteration: the flow (Navier-Stokes, Newton) with the last T, then the energy equation (linear) with that u"""
    count = 0
    def __init__(self, cv, relax=1.0):
        collect(); Picard.count += 1
        self.cv = cv; msh = cv.msh; c = msh.basix_cell(); d = cv.deg; self.relax = relax
        Wf = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, d, shape=(2,)), basix.ufl.element("Lagrange", c, d - 1)]))
        self.wf = fem.Function(Wf); self.T = fem.Function(cv.S)
        u, p = ufl.split(self.wf); v, q = ufl.TestFunctions(Wf)
        Ff = (ufl.inner(ufl.grad(u) * u, v) + cv.Pr * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)
              - cv.Pr * cv.Ra * self.T * ufl.dot(cv.ey, v)) * ufl.dx
        V, _ = Wf.sub(0).collapse(); Q, _ = Wf.sub(1).collapse()
        allb = mesh.locate_entities_boundary(msh, 1, lambda x: np.full(x.shape[1], True))
        bf = [fem.dirichletbc(fem.Function(V), fem.locate_dofs_topological((Wf.sub(0), V), 1, allb), Wf.sub(0)),
              fem.dirichletbc(fem.Function(Q), fem.locate_dofs_geometrical((Wf.sub(1), Q), lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)), Wf.sub(1))]
        self.flow = NonlinearProblem(Ff, self.wf, bcs=bf, petsc_options_prefix=f"pf{Picard.count}_", petsc_options=NEWTON)
        Tt = ufl.TrialFunction(cv.S); s = ufl.TestFunction(cv.S); uu = ufl.split(self.wf)[0]
        self.Tn = fem.Function(cv.S); Th = fem.Function(cv.S); Th.x.array[:] = 1.0
        bT = [fem.dirichletbc(Th, fem.locate_dofs_topological(cv.S, 1, cv.hot)), fem.dirichletbc(fem.Function(cv.S), fem.locate_dofs_topological(cv.S, 1, cv.cold))]
        self.energy = fp.LinearProblem((ufl.dot(uu, ufl.grad(Tt)) * s + ufl.dot(ufl.grad(Tt), ufl.grad(s))) * ufl.dx, fem.Constant(msh, 0.0) * s * ufl.dx,
                                       bcs=bT, u=self.Tn, petsc_options_prefix=f"pe{Picard.count}_", petsc_options=LU)
        self.T.x.array[:] = cv.w.x.array[cv.mapS]                       # start from the state of cv

    def run(self, Ra, tol=1e-8, maxit=200):
        """returns (iterations, history of max|T_k - T_(k-1)|); an inf at the end: the flow solve failed"""
        self.cv.Ra.value = Ra; hist = []
        for k in range(maxit):
            self.flow.solve()
            if self.flow.solver.getConvergedReason() <= 0:
                return k + 1, hist + [np.inf]
            self.energy.solve()
            dT = comm.allreduce(np.abs(self.Tn.x.array - self.T.x.array).max(), op=MPI.MAX); hist.append(dT)
            self.T.x.array[:] = self.relax * self.Tn.x.array + (1 - self.relax) * self.T.x.array
            if dT < tol or not np.isfinite(dT) or dT > 1e3:
                return k + 1, hist
        return maxit, hist

# de Vahl Davis (1983), Pr = 0.71, scale kappa/L: mean Nu, u_max on x = 0.5 (value, y), v_max on y = 0.5 (value, x)
DVD = {1e3: dict(Nu=1.118, umax=(3.649, 0.813), vmax=(3.697, 0.178)), 1e4: dict(Nu=2.243, umax=(16.178, 0.823), vmax=(19.617, 0.119)),
       1e5: dict(Nu=4.519, umax=(34.73, 0.855), vmax=(68.59, 0.066)), 1e6: dict(Nu=8.800, umax=(64.63, 0.850), vmax=(219.36, 0.0379))}

## 2 · Scales before code  *(lecture §A2–A3)*
The thermal boundary layer on the vertical walls is $\delta_T\sim L\,Ra^{-1/4}$. How many cells of a uniform and of a graded mesh (`grade=0.5`: the spacing at the wall is $(1-0.5)\,h$) lie inside it?

In [ ]:
import numpy as np
g = 0.5
for Ra in (1e3, 1e4, 1e5, 1e6):
    d = Ra ** -0.25
    cells = []
    for N in (16, 32, 64):
        s = np.linspace(0, 1, N + 1); xg = s - g * np.sin(2 * np.pi * s) / (2 * np.pi)
        cells += [int((s < d).sum() - 1), int((xg < d).sum() - 1)]
    print(f"Ra {Ra:.0e}: delta_T ~ Ra^-1/4 = {d:.4f};  cells inside (uniform/graded) N = 16: {cells[0]}/{cells[1]}, N = 32: {cells[2]}/{cells[3]}, N = 64: {cells[4]}/{cells[5]}")

> **✏️ Your turn (3 min).** (i) Which meshes resolve the boundary layer at $Ra=10^6$ with at least two P2 cells? (ii) The benchmark velocities are in units of $\kappa/L$. A code written with $\nu/L$ reports
> velocities larger or smaller by which factor, and does the Nusselt number change?

> *your answer*

## 3 · The cavity at $Ra=10^3$: two ways to compute $Nu$  *(lecture §B1–B3, §C2)*
Hot wall $x=0$ ($T=1$), cold wall $x=1$ ($T=0$), adiabatic top and bottom, no slip, $Pr=0.71$. Newton from the conduction state. `nusselt()` integrates $-\partial T/\partial x$ on the wall;
`nusselt_flux()` evaluates the weak energy equation with a test function that is 1 on the wall (the consistent flux).

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from bouss import *
DVD = {1e3: dict(Nu=1.118, umax=(3.649, 0.813), vmax=(3.697, 0.178)), 1e4: dict(Nu=2.243, umax=(16.178, 0.823), vmax=(19.617, 0.119)),
       1e5: dict(Nu=4.519, umax=(34.73, 0.855), vmax=(68.59, 0.066)), 1e6: dict(Nu=8.800, umax=(64.63, 0.850), vmax=(219.36, 0.0379))}

cv = Convection(box_mesh(32, grade=0.5)); r = cv.solve(1e3)
say(f"Ra 1e3: SNES reason {r[0]}, {r[1]} Newton iterations;  residuals {np.array2string(cv.history(), precision=2)}")
say(f"Nu wall gradient (hot/cold) {cv.nusselt()[0]:.6f}/{cv.nusselt()[1]:.6f};  consistent flux {cv.nusselt_flux()[0]:.6f}/{cv.nusselt_flux()[1]:.6f};  de Vahl Davis {DVD[1e3]['Nu']}")
um, vm = cv.peak(0, x0=0.5), cv.peak(1, y0=0.5)
say(f"u_max on x = 0.5: {um[0]:.4f} at y = {um[1]:.4f} (benchmark {DVD[1e3]['umax']});  v_max on y = 0.5: {vm[0]:.4f} at x = {vm[1]:.4f} (benchmark {DVD[1e3]['vmax']})")
t, Tm = cv.line(2, y0=0.5); _, vmid = cv.line(1, y0=0.5); _, umid = cv.line(0, x0=0.5)
if comm.rank == 0:
    np.savez("/tmp/fnx/ra1e3.npz", t=t, T=Tm, v=vmid, u=umid)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load("/tmp/fnx/ra1e3.npz"); t = d["t"]
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].plot(t, d["T"]); ax[0].plot(t, 1 - t, "k--", lw=0.8, label="conduction"); ax[0].set(xlabel="x", ylabel="T on y = 0.5"); ax[0].legend()
ax[1].plot(t, d["v"]); ax[1].set(xlabel="x", ylabel="v on y = 0.5"); ax[2].plot(d["u"], t); ax[2].set(xlabel="u on x = 0.5", ylabel="y")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout(); plt.show()

> **✏️ Your turn (5 min).** (i) Newton needed few iterations from the conduction state. Why is $Ra=10^3$ easy? (ii) Both Nusselt numbers agree with the benchmark here. Which one would you trust on a coarser mesh, and why?
> (iii) The hot and cold Nusselt numbers agree to all digits. Is that a check of energy conservation?

> *your answer*

## 4 · Up to $Ra=10^6$: continuation or a direct start  *(lecture §B4)*
Continuation: each Newton solve starts from the solution at the previous $Ra$. Then a direct start from the conduction state at each $Ra$ (no line search, as in the module).

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from bouss import *
DVD = {1e3: dict(Nu=1.118, umax=(3.649, 0.813), vmax=(3.697, 0.178)), 1e4: dict(Nu=2.243, umax=(16.178, 0.823), vmax=(19.617, 0.119)),
       1e5: dict(Nu=4.519, umax=(34.73, 0.855), vmax=(68.59, 0.066)), 1e6: dict(Nu=8.800, umax=(64.63, 0.850), vmax=(219.36, 0.0379))}

cv = Convection(box_mesh(32, grade=0.5))
for Ra in (1e3, 1e4, 1e5, 1e6):
    r = cv.solve(Ra); um, vm = cv.peak(0, x0=0.5), cv.peak(1, y0=0.5); b = DVD[Ra]
    say(f"Ra {Ra:.0e}: reason {r[0]}, {r[1]} its | Nu flux {cv.nusselt_flux()[0]:.5f}, gradient {cv.nusselt()[0]:.5f} (DVD {b['Nu']}) | u_max {um[0]:.3f} at {um[1]:.4f} ({b['umax'][0]} at {b['umax'][1]}) | v_max {vm[0]:.3f} at {vm[1]:.4f} ({b['vmax'][0]} at {b['vmax'][1]})")
for Ra in (1e4, 1e5, 1e6):
    cd = Convection(box_mesh(32, grade=0.5)); r = cd.solve(Ra)
    say(f"direct from conduction, Ra {Ra:.0e}: SNES reason {r[0]}, {r[1]} iterations, Nu flux {cd.nusselt_flux()[0]:.4g}")

> **✏️ Your turn (5 min).** (i) Where do our values and de Vahl Davis (1983) differ most, and which do you believe (lecture §C3)? (ii) Why does the direct start fail at $Ra=10^5$ — the solution exists, Newton is the same?
> (iii) Turn on the line search (`NEWTON["snes_linesearch_type"] = "bt"` before creating the problem) and repeat the direct start at $Ra=10^6$. Is a converged direct start a reason to drop continuation?

> *your answer*

## 5 · Picard (segregated) against Newton (monolithic)  *(lecture §B5)*
The segregated iteration solves the flow with the last temperature, then the temperature with that flow, and repeats. On a coarse mesh ($N=16$) at three Rayleigh numbers:

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from bouss import *

for Ra in (1e3, 1e4, 3e4):
    cv = Convection(box_mesh(16, grade=0.5)); pc = Picard(cv); k, h = pc.run(Ra, maxit=30)
    q = np.array(h[2:8]) / np.array(h[1:7])
    say(f"Picard Ra {Ra:.0e}: {k} iterations, last change {h[-1]:.1e}, contraction factors {np.array2string(q, precision=3)}")
    cn = Convection(box_mesh(16, grade=0.5)); r = cn.solve(Ra) if Ra < 2e4 else cn.continuation([1e4, Ra])[-1][1:]
    say(f"     Newton: reason {r[0]}, {r[1]} iterations")

> **✏️ Your turn (5 min).** (i) Estimate how many Picard iterations a tolerance of $10^{-8}$ needs at $Ra=10^4$ from the contraction factor. (ii) What does Picard leave out of the Jacobian, and why does that matter more at high $Ra$?

> *your answer*

## 6 · LLM exercise: the cavity at $Ra=10^5$  *(lecture §D3)*
Ask an LLM for a DOLFINx 0.11 script that solves the de Vahl Davis cavity at $Ra=10^5$, $Pr=0.71$, and prints the mean Nusselt number, $u_{\max}$ on $x=0.5$ and $v_{\max}$ on $y=0.5$ with their positions,
**compared with the benchmark**. Paste it below, run it with `-np 2`, and judge it against the reference cell.

In [ ]:
%%fenicsx -np 2
# Paste the LLM's script here (keep the %%fenicsx line). Then run the reference cell below.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from bouss import *
DVD = {1e3: dict(Nu=1.118, umax=(3.649, 0.813), vmax=(3.697, 0.178)), 1e4: dict(Nu=2.243, umax=(16.178, 0.823), vmax=(19.617, 0.119)),
       1e5: dict(Nu=4.519, umax=(34.73, 0.855), vmax=(68.59, 0.066)), 1e6: dict(Nu=8.800, umax=(64.63, 0.850), vmax=(219.36, 0.0379))}

cv = Convection(box_mesh(32, grade=0.5)); its = cv.continuation([1e3, 1e4, 1e5])
um, vm = cv.peak(0, x0=0.5), cv.peak(1, y0=0.5); b = DVD[1e5]
say(f"continuation {[(f'{a:.0e}', r, k) for a, r, k in its]}")
say(f"Ra 1e5: Nu {cv.nusselt_flux()[0]:.5f} (DVD {b['Nu']}), u_max {um[0]:.3f} at y = {um[1]:.4f} ({b['umax']}), v_max {vm[0]:.3f} at x = {vm[1]:.4f} ({b['vmax']})")

> *your verdict on the LLM's code*